In [1]:
import pandas as pd 
import json
import mne
import numpy as np
import mne
import os
from itertools import product
import glob
import keras
from keras import layers, models, Model, ops


# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [2]:
tsv_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/participants.tsv"

df = pd.read_csv(tsv_path, sep="\t")
print(df.head())

  participant_id GROUP    ID     EEG  AGE GENDER  MOCA  UPDRS  TYPE
0        sub-001    PD  1001  PD1001   80      M    19   28.0     1
1        sub-002    PD  1011  PD1011   81      M    17   25.0     1
2        sub-003    PD  1021  PD1021   68      F    26   10.0     1
3        sub-004    PD  1031  PD1031   80      M    22   10.0     1
4        sub-005    PD  1041  PD1041   56      M    21   13.0     1


In [3]:
set_file_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.set"

# Load the raw EEG data using MNE
raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
eeg_signals = raw.get_data()

print("Shape of EEG signals array (C, L):", eeg_signals.shape)

Reading /kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
Shape of EEG signals array (C, L): (63, 140830)


/tmp/ipykernel_58/2417030768.py:4: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True)


In [4]:
channel_names = raw.ch_names
print(f"Total number of channels: {len(channel_names)}")
print("Channel names:")
print(", ".join([f"{i+1}: {ch}" for i, ch in enumerate(channel_names)]))

Total number of channels: 63
Channel names:
1: Fp1, 2: Fz, 3: F3, 4: F7, 5: FT9, 6: FC5, 7: FC1, 8: C3, 9: T7, 10: TP9, 11: CP5, 12: CP1, 13: P3, 14: P7, 15: O1, 16: Oz, 17: O2, 18: P4, 19: P8, 20: TP10, 21: CP6, 22: CP2, 23: Cz, 24: C4, 25: T8, 26: FT10, 27: FC6, 28: FC2, 29: F4, 30: F8, 31: Fp2, 32: AF7, 33: AF3, 34: AFz, 35: F1, 36: F5, 37: FT7, 38: FC3, 39: C1, 40: C5, 41: TP7, 42: CP3, 43: P1, 44: P5, 45: PO7, 46: PO3, 47: POz, 48: PO4, 49: PO8, 50: P6, 51: P2, 52: CPz, 53: CP4, 54: TP8, 55: C6, 56: C2, 57: FC4, 58: FT8, 59: F6, 60: AF8, 61: AF4, 62: F2, 63: FCz


In [5]:
mne.set_log_level('ERROR')

base_dir = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset"
common_channels = None

for sub_id in range(1, 150):
    sub_str = f"sub-{sub_id:03d}"
    set_file_path = os.path.join(base_dir, sub_str, "eeg", f"{sub_str}_task-Rest_eeg.set")
    
    if os.path.exists(set_file_path):
        raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
        raw.rename_channels({ch: ch.strip() for ch in raw.ch_names})
        raw.pick("eeg")
        ch_set = set(raw.ch_names)
        
        if common_channels is None:
            common_channels = ch_set
        else:
            common_channels = common_channels.intersection(ch_set)

# Reset MNE log level back to default if desired
mne.set_log_level('INFO')

common_channels_list = sorted(list(common_channels))
print(f"Total common EEG channels across all subjects: {len(common_channels_list)}")
print("Common channels:")
print(", ".join(common_channels_list))


/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, pr

Total common EEG channels across all subjects: 60
Common channels:
AF3, AF4, AF7, AF8, AFz, C1, C2, C3, C4, C5, C6, CP1, CP2, CP3, CP4, CP5, CP6, CPz, Cz, F1, F2, F3, F4, F5, F6, F7, F8, FC1, FC2, FC3, FC4, FC5, FC6, FCz, FT10, FT7, FT8, Fp1, Fp2, Fz, O1, O2, Oz, P1, P2, P3, P4, P5, P6, P7, P8, PO7, PO8, POz, T7, T8, TP10, TP7, TP8, TP9


/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_58/1694620427.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)


In [6]:
def load_segment_set(set_file_path, l_freq, h_freq, target_sfreq=256, window_sec=2, 
                     overlap_ratio=0.5, peak_to_peak_threshold=0.00028, snr_db=None):
    
    # Load recording (using read_raw_eeglab for .set/.fdt files)
    raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)

    # Define the precise channel order requested
    target_channels = common_channels_list

    # Reorder and pick the specific channels
    valid_channels = [ch for ch in target_channels if ch in raw.ch_names]
    raw.pick(valid_channels)

    # Bandpass Filter (0.5 to 45 Hz)
    raw.filter(l_freq=0.5, h_freq=45.0, fir_design='firwin', verbose=False)

    # Notch Filter at 50 Hz to eliminate line noise
    raw.notch_filter(freqs=50.0, fir_design='firwin', verbose=False)

    # Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', verbose=False)

    # Resample to target frequency
    raw.resample(target_sfreq, verbose=False)

    # Get data matrix
    signals = raw.get_data()
    
    # Add Gaussian Noise if snr_db is provided
    if snr_db is not None:
        # Calculate signal power across all channels
        signal_power = np.mean(signals ** 2)
        
        # Convert dB SNR to linear ratio: SNR_linear = 10^(SNR_dB / 10)
        snr_linear = 10 ** (snr_db / 10.0)
        
        # Calculate required noise power: P_noise = P_signal / SNR_linear
        noise_power = signal_power / snr_linear
        noise_std = np.sqrt(noise_power)
        
        # Generate white Gaussian noise and add to signals
        noise = np.random.normal(0, noise_std, size=signals.shape)
        signals = signals + noise

    print("Signal shape (C, L):", signals.shape)
    C, L = signals.shape
    window_samples = int(window_sec * target_sfreq)

    # Calculate stride samples based on the overlap ratio
    stride_samples = int(window_samples * (1 - overlap_ratio))
    if stride_samples < 1:
        stride_samples = 1

    # Generate sequential windows & Apply Artifact Rejection
    window_list = []
    start = 0
    while start + window_samples <= L:
        end = start + window_samples
        window = signals[:, start:end]
        
        # Peak-to-Peak Threshold Artifact Rejection
        peak_to_peak = np.ptp(window, axis=1)
        if np.any(peak_to_peak > peak_to_peak_threshold):
            start += stride_samples
            continue
        
        window_list.append(window)
        start += stride_samples

    # Check if any windows were created
    if len(window_list) == 0:
        return np.empty((0, C, window_samples))

    # Convert to standard array format (N, C, T)
    windows = np.array(window_list)
    if l_freq > 0 and h_freq > 0:
        windows = mne.filter.filter_data(data=windows, sfreq=target_sfreq, l_freq=l_freq, h_freq=h_freq, method='iir', verbose=False)

    return windows

In [7]:
ids = df.iloc[:,0].values
state = df.iloc[:,1].values 

In [8]:
def get_data(l_freq, h_freq, peak_to_peak_threshold=0.00028):
    X_pd = []
    X_hc = []
    
    # Base directory path for the dataset
    base_dir = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset"
    
    # Loop through subject indices from 1 to 149
    for sub_id in range(1, 150):
        print("patient number is", sub_id)
        sub_str = f"sub-{sub_id:03d}"
        set_file_path = os.path.join(base_dir, sub_str, "eeg", f"{sub_str}_task-Rest_eeg.set")
        
        # Check if file exists before attempting to load
        if not os.path.exists(set_file_path):
            print(f"File not found for subject {sub_id}")
            continue
            
        if sub_id < 101:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0.0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_pd.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
        else:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_hc.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
    return X_hc, X_pd

In [9]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [10]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [11]:
import keras
from keras import layers, models, Model, ops

# --- Custom Layers for EEG Conformer ---

class RearrangeToSeq(layers.Layer):
    """Reshapes (Batch, 1, Time, Filters) -> (Batch, Time, Filters) for Transformer inputs."""
    def call(self, x):
        return ops.squeeze(x, axis=1)


class MultiHeadSelfAttention(layers.Layer):
    """Multi-Head Self-Attention using scaled dot-product compatible with Keras 3."""
    def __init__(self, emb_size=40, num_heads=10, dropout=0.5, **kwargs):
        super().__init__(**kwargs)
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_dim = emb_size // num_heads

        self.q_dense = layers.Dense(emb_size)
        self.k_dense = layers.Dense(emb_size)
        self.v_dense = layers.Dense(emb_size)
        self.out_dense = layers.Dense(emb_size)
        self.att_drop = layers.Dropout(dropout)

    def split_heads(self, x, batch_size):
        x = ops.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return ops.transpose(x, axes=[0, 2, 1, 3])

    def call(self, x, training=False):
        batch_size = ops.shape(x)[0]

        q = self.split_heads(self.q_dense(x), batch_size)
        k = self.split_heads(self.k_dense(x), batch_size)
        v = self.split_heads(self.v_dense(x), batch_size)

        matmul_qk = ops.matmul(q, ops.transpose(k, axes=[0, 1, 3, 2]))
        dk = ops.cast(self.head_dim, "float32")
        scaled_attention_logits = matmul_qk / ops.sqrt(dk)

        attention_weights = ops.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.att_drop(attention_weights, training=training)

        output = ops.matmul(attention_weights, v)
        output = ops.transpose(output, axes=[0, 2, 1, 3])
        concat_attention = ops.reshape(output, (batch_size, -1, self.emb_size))

        return self.out_dense(concat_attention)


class TransformerEncoderBlock(layers.Layer):
    """Standard Transformer Encoder Block with Pre-LayerNormalization."""
    def __init__(self, emb_size=40, num_heads=10, forward_expansion=4, drop_p=0.5, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = MultiHeadSelfAttention(emb_size=emb_size, num_heads=num_heads, dropout=drop_p)
        self.drop1 = layers.Dropout(drop_p)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = models.Sequential([
            layers.Dense(emb_size * forward_expansion, activation='gelu'),
            layers.Dropout(drop_p),
            layers.Dense(emb_size),
            layers.Dropout(drop_p)
        ])

    def call(self, x, training=False):
        res1 = x
        x_norm1 = self.norm1(x)
        attn_out = self.attn(x_norm1, training=training)
        x = res1 + self.drop1(attn_out, training=training)

        res2 = x
        x_norm2 = self.norm2(x)
        ffn_out = self.ffn(x_norm2, training=training)
        x = res2 + ffn_out

        return x


def create_eeg_conformer(
    input_shape=(22, 1000),
    nb_classes=4,
    emb_size=40,
    depth=6,
    num_heads=10
):
    inputs = layers.Input(shape=input_shape)

    if len(input_shape) == 2:
        x = layers.Reshape((1, input_shape[0], input_shape[1]))(inputs)
    else:
        x = inputs

    x = layers.Permute((2, 3, 1))(x)

    x = layers.Conv2D(emb_size, kernel_size=(1, 25), strides=(1, 1), padding='same', use_bias=True)(x)
    n_chans = input_shape[0] if len(input_shape) == 2 else input_shape[1]
    x = layers.Conv2D(emb_size, kernel_size=(n_chans, 1), strides=(1, 1), padding='valid', use_bias=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)

    x = layers.AveragePooling2D(pool_size=(1, 75), strides=(1, 15))(x)
    x = layers.Dropout(0.5)(x)

    x = RearrangeToSeq()(x)

    for _ in range(depth):
        x = TransformerEncoderBlock(emb_size=emb_size, num_heads=num_heads, drop_p=0.5)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='elu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='elu')(x)
    x = layers.Dropout(0.3)(x)

    activation = 'sigmoid' if nb_classes == 1 else 'softmax'
    outputs = layers.Dense(nb_classes, activation=activation)(x)

    model = Model(inputs=inputs, outputs=outputs, name="EEG_Conformer")
    return model

In [12]:
def format_eeg_tensor_conformer(data_array):
    """
    Formats input EEG data array into standard 3D matrix for EEG Conformer:
    (Batch/Epochs, Channels, Timepoints).
    """
    arr = np.asarray(data_array, dtype=np.float32)

    if arr.ndim == 2:
        # Single Epoch (Channels, Time) -> (1, Channels, Time)
        return np.expand_dims(arr, axis=0)
    elif arr.ndim == 3:
        # Check if shape is (Epochs, Time, Channels) and transpose if necessary
        # Conformer expects (Epochs, Channels, Time)
        if arr.shape[1] > arr.shape[2]:  # If Time > Channels in axis 1
            return np.transpose(arr, (0, 2, 1))
        return arr
    elif arr.ndim == 4:
        # Remove singleton dimensions e.g. (Epochs, 1, Channels, Time)
        arr = np.squeeze(arr)
        if arr.ndim == 2:
            return np.expand_dims(arr, axis=0)
        elif arr.shape[1] > arr.shape[2]:
            return np.transpose(arr, (0, 2, 1))
        return arr
    else:
        raise ValueError(f"Unexpected array dimension: {arr.ndim} (shape: {arr.shape})")


def run_subject_level_mc_cv_conformer(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    # Infer input shape from single epoch: (Channels, Timepoints)
    sample_sub = X_healthy[0]
    sample_epoch = sample_sub[0] if sample_sub.ndim == 3 else sample_sub

    if sample_epoch.ndim == 2:
        # Ensure (Channels, Timepoints) order
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    elif sample_epoch.ndim == 3:
        sample_epoch = np.squeeze(sample_epoch)
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    else:
        raise ValueError(f"Unexpected epoch shape: {sample_epoch.shape}")

    print(f"--> Inferred EEG Conformer Input Shape (Channels, Timepoints): {input_shape}")

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # EEG Conformer Hyperparameter Grid Search
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'emb_size': [40],
        'depth': [6],
        'num_heads': [10]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None
        best_threshold = 75

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []

            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

                # Format to 3D (Batch, Channels, Timepoints)
                X_inner_train = format_eeg_tensor_conformer(X_inner_train)

                # Shuffle training data
                shuffle_idx = np.random.RandomState(SEED).permutation(len(X_inner_train))
                X_inner_train = X_inner_train[shuffle_idx]
                y_inner_train = y_inner_train[shuffle_idx]

                # Train/Val split
                val_size = int(len(X_inner_train) * 0.1)
                X_tr, y_tr = X_inner_train[val_size:], y_inner_train[val_size:]
                X_va, y_va = X_inner_train[:val_size], y_inner_train[:val_size]

                # Instantiate EEG Conformer Model
                inner_model = create_eeg_conformer(
                    input_shape=input_shape,
                    nb_classes=1,
                    emb_size=params['emb_size'],
                    depth=params['depth'],
                    num_heads=params['num_heads']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )

                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                inner_model.fit(
                    X_tr, y_tr,
                    epochs=40, batch_size=params['batch_size'],
                    verbose=1, validation_data=(X_va, y_va), callbacks=[early_stop]
                )

                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_subject_ratios = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = format_eeg_tensor_conformer(sub)

                    if sub_array.shape[0] == 0:
                        continue

                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t

                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))

        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")

        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)

        X_train_final = format_eeg_tensor_conformer(X_train_final)

        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_train_final))
        X_train_final = X_train_final[shuffle_idx_final]
        y_train_final = y_train_final[shuffle_idx_final]

        val_size_final = int(len(X_train_final) * 0.1)
        X_tr_f, y_tr_f = X_train_final[val_size_final:], y_train_final[val_size_final:]
        X_va_f, y_va_f = X_train_final[:val_size_final], y_train_final[:val_size_final]

        final_model = create_eeg_conformer(
            input_shape=input_shape,
            nb_classes=1,
            emb_size=best_params['emb_size'],
            depth=best_params['depth'],
            num_heads=best_params['num_heads']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
        final_model.fit(
            X_tr_f, y_tr_f,
            epochs=80, batch_size=best_params['batch_size'],
            verbose=1, validation_data=(X_va_f, y_va_f), callbacks=[early_stop_final]
        )

        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = format_eeg_tensor_conformer(sub)

            if sub_array.shape[0] == 0:
                continue

            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)

            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}%")

    summary_df = pd.DataFrame(fold_summary_records)
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy: {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [13]:
X_hc,X_pd = get_data(8,12)
df = run_subject_level_mc_cv_conformer(X_hc, X_pd, SEED=42)
print(df)

patient number is 1


/tmp/ipykernel_58/1350230097.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 72105)
(59, 60, 512)
patient number is 2


/tmp/ipykernel_58/1350230097.py:5: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 83466)
(163, 60, 512)
patient number is 3


...

166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.5034 - loss: 1.2894

2026-08-28 18:15:55.080864: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


166/166 ━━━━━━━━━━━━━━━━━━━━ 36s 105ms/step - accuracy: 0.4994 - loss: 0.9875 - val_accuracy: 0.5700 - val_loss: 0.6800
Epoch 2/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 76ms/step - accuracy: 0.5405 - loss: 0.7400 - val_accuracy: 0.6621 - val_loss: 0.7736
Epoch 3/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 77ms/step - accuracy: 0.6401 - loss: 0.6452 - val_accuracy: 0.7543 - val_loss: 0.5987
Epoch 4/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 77ms/step - accuracy: 0.7425 - loss: 0.5309 - val_accuracy: 0.7918 - val_loss: 0.6415
Epoch 5/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 76ms/step - accuracy: 0.7991 - loss: 0.4459 - val_accuracy: 0.8294 - val_loss: 0.4286
Epoch 6/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 76ms/step - accuracy: 0.6884 - loss: 0.5858 - val_accuracy: 0.7526 - val_loss: 0.7672
Epoch 7/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.8232 - loss: 0.4049 - val_accuracy: 0.8823 - val_loss: 0.3498
Epoch 8/80
166/166 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - accuracy: 0.8624 - loss: 0.3369 - val_accura

2026-08-28 18:26:09.927235: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-08-28 18:26:15.041184: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 5/9 | PD: 14/19 | Acc: 67.86%

Total Combined Correct: 90/148
Overall Nested Cross-Validation Accuracy: 60.81%

--- Nested Cross-Validation Summary ---
 Fold Number                                                          Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65            5/10      17/20            73.33%         22/30
           2 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65            6/10      11/20            56.67%         17/30
           3 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     75            7/10       9/20            53.33%         16/30
           4 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65            5/10 

link : https://github.com/eeyhsong/EEG-Conformer/blob/main/conformer.py